Data quality gates (Day 23).
Reads:  jobmarket.silver.silver_job_postings
Runs BETWEEN merge_to_silver and extract_skills in the pipeline.
Fail-fast: any assertion failure fails the task and blocks Gold.
Every check prints its numbers -> the run log is a daily health report.

In [0]:
from pyspark.sql import functions as F

sp = spark.table("jobmarket.silver.silver_job_postings")

# collect all failures, raise once at the end -> the log shows EVERY
# problem, not just the first (a partial health report is less useful)
failures = []

# --- Gate 1: row count in a sane band ---
# Justification: Silver held 117,292 at Day 10; it only grows (Adzuna
# appends daily, Kaggle is fixed). A count BELOW ~117k means the Kaggle
# backfill vanished or MERGE broke. No hard upper bound (it grows), but
# a suspiciously huge jump would suggest dedup failed.
row_count = sp.count()
MIN_ROWS = 200_000
print(f"[Gate 1] Silver row count: {row_count:,} (floor: {MIN_ROWS:,})")
if row_count < MIN_ROWS:
    failures.append(f"Row count {row_count:,} below floor {MIN_ROWS:,}")

In [0]:
# --- Gate 2: required fields must be ~fully populated ---
# Justification: posting_id, title_norm, posted_date are the spine —
# nulls here mean parsing or key-generation broke. Allow a tiny
# tolerance (0.1%) for genuine edge cases, not zero-tolerance brittleness.
required = ["posting_id", "title_norm", "posted_date"]
for c in required:
    null_pct = sp.filter(F.col(c).isNull()).count() / row_count * 100
    print(f"[Gate 2] {c} null rate: {null_pct:.2f}%")
    if null_pct > 0.1:
        failures.append(f"{c} null rate {null_pct:.2f}% exceeds 0.1%")

# --- Gate 3: salary null-rate DRIFT detector ---
# Justification (the clever one): salary is ~76% null and that's FINE —
# but a sudden jump to ~95% means the salary parser broke or the source
# schema drifted. We gate on a BAND around the known value, not a
# fixed ceiling. The Day 4 schema-drift lesson, as a live monitor.
salary_null_pct = sp.filter(F.col("salary_min").isNull()).count() / row_count * 100
SALARY_NULL_LOW, SALARY_NULL_HIGH = 60.0, 85.0
print(f"[Gate 3] salary_min null rate: {salary_null_pct:.1f}% "
      f"(expected band: {SALARY_NULL_LOW}–{SALARY_NULL_HIGH}%)")
if not (SALARY_NULL_LOW <= salary_null_pct <= SALARY_NULL_HIGH):
    failures.append(
        f"salary null rate {salary_null_pct:.1f}% outside "
        f"[{SALARY_NULL_LOW}, {SALARY_NULL_HIGH}] — parser break or schema drift?")

In [0]:
# --- Gate 4: no duplicate posting_id ---
# Justification: posting_id is the deterministic dedup key. Duplicates
# here mean dedup or MERGE broke its core contract — and would
# double-count everything downstream. Must be exactly zero.
distinct_ids = sp.select("posting_id").distinct().count()
dupes = row_count - distinct_ids
print(f"[Gate 4] posting_id duplicates: {dupes} (must be 0)")
if dupes > 0:
    failures.append(f"{dupes} duplicate posting_ids — dedup/MERGE contract broken")

# --- The gate: raise once, with the full report ---
print("\n" + "="*50)
if failures:
    print(f"❌ DATA QUALITY FAILED — {len(failures)} issue(s):")
    for f in failures:
        print(f"   • {f}")
    raise Exception(f"Data quality gate failed: {len(failures)} issue(s) — see log")
print("✅ ALL DATA QUALITY GATES PASSED")